In [ ]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
from torch.utils.data import DataLoader

#from data_scripts.data_import import BaseballVideos  # <-- your file

In [ ]:
!pwd

/content


## Pretrained Neural Network

In [ ]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
from torch.utils.data import DataLoader

#from data_scripts.data_import import BaseballVideos  # <-- your file

In [ ]:
import os
import numpy as np
from xml.dom import minidom
import torch
from torch.utils.data import Dataset
from torchvision import tv_tensors
from PIL import Image
import cv2 as cv

class BaseballVideos(Dataset):
    def __init__(self, root=None, transforms=None):
        self.root = root if root is not None else os.getcwd()
        self.transforms = transforms

        # Gather video and annotation files
        self.vids = sorted([f for f in os.listdir(self.root) if f.endswith(".mov")])
        self.notes = sorted([f for f in os.listdir(self.root) if f.endswith(".xml")])

        # Build a list of (video_path, xml_path, frame_number)
        self.frame_index = []
        for vid, note in zip(self.vids, self.notes):
            vid_path = os.path.join(self.root, vid)
            note_path = os.path.join(self.root, note)
            cap = cv.VideoCapture(vid_path)
            frame_count = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
            for f in range(frame_count):
                self.frame_index.append((vid_path, note_path, f))
            cap.release()

    def __len__(self):
        return len(self.frame_index)

    def __getitem__(self, idx):
        vid_path, note_path, frame_num = self.frame_index[idx]

        # Load the frame on-the-fly
        cap = cv.VideoCapture(vid_path)
        cap.set(cv.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        cap.release()
        if not ret:
            raise RuntimeError(f"Could not read frame {frame_num} from {vid_path}")

        # Convert frame to PIL Image (cv2 loads as BGR, PIL expects RGB)
        frame_rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
        img = Image.fromarray(frame_rgb)

        # Parse XML for this frame
        note = minidom.parse(note_path)
        frame_i = [j for j in note.getElementsByTagName("box") if int(j.attributes['frame'].value) == frame_num]
        boxes, labels, areas, movings = [], [], [], []

        canvas_size = [img.height, img.width]
        for j in frame_i:
            moving = j.getElementsByTagName('attribute')[0].firstChild.data == 'true'
            xtl = float(j.attributes['xtl'].value)
            ytl = float(j.attributes['ytl'].value)
            xbr = float(j.attributes['xbr'].value)
            ybr = float(j.attributes['ybr'].value)
            box = (xtl, ytl, xbr, ybr)
            label = 'baseball'
            area = (xbr - xtl) * (ybr - ytl)
            boxes.append(box)
            labels.append(label)
            areas.append(area)
            movings.append(moving)

        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=canvas_size),
            "labels": labels,
            "area": areas,
            "moving": movings
        }

        if self.transforms is not None:
            img = self.transforms(img)  # Only image is transformed here

        return img, target

In [ ]:
# pretrained model
import torchvision.models as models
import torch.nn as nn

class BaseballTrackerPretrained(nn.Module):
    def __init__(self, num_outputs=4):
        super().__init__()
        # Load pretrained ResNet18
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Remove the final classification layer
        self.backbone = nn.Sequential(*list(self.backbone.children())[:-1])  # Output: (batch, 512, 1, 1)
        self.flatten = nn.Flatten()
        self.fc_bbox = nn.Linear(512, num_outputs)  # For bounding box regression
        self.fc_move = nn.Linear(512, 1)            # For movement classification

    def forward(self, x):
        x = self.backbone(x)
        x = self.flatten(x)
        bbox = self.fc_bbox(x)
        is_moving = torch.sigmoid(self.fc_move(x))
        return bbox, is_moving

In [ ]:
import torchvision.transforms as T

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
def collate_fn(batch):
    imgs, bboxes, movings = [], [], []
    for img, target in batch:
        # Skip frames with no baseballs
        if len(target["boxes"]) == 0 or len(target["moving"]) == 0:
            continue
        # Apply transform (handles PIL Image to tensor and normalization)
        img = transform(img)
        # Use the first bounding box and movement label
        box = target["boxes"][0]
        moving = float(target["moving"][0])
        imgs.append(img)
        bboxes.append(torch.tensor(box, dtype=torch.float32))
        movings.append(torch.tensor(moving, dtype=torch.float32))
    if len(imgs) == 0:
        # Return empty tensors if all frames are skipped
        return torch.empty(0), torch.empty(0), torch.empty(0)
    imgs = torch.stack(imgs)
    bboxes = torch.stack(bboxes)
    movings = torch.stack(movings)
    return imgs, bboxes, movings

In [ ]:
# --- Training Loop ---
def train(train_loader, val_loader):
    model = BaseballTrackerPretrained()
    #model.load_state_dict(torch.load("baseball_tracker_pretrained.pth")) # ADDED FOR 2ND ROUND OF TRAINING
    # model.train()  # Set to training mode # ADDED FOR 2ND ROUND OF TRAINING

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_bbox_fn = nn.SmoothL1Loss()
    loss_move_fn = nn.BCELoss()

    for epoch in range(0,20):
        model.train()
        for images, bboxes, is_moving in train_loader:
            if images.size(0) == 0:
                continue
            pred_bbox, pred_move = model(images)
            loss_bbox = loss_bbox_fn(pred_bbox, bboxes)
            loss_move = loss_move_fn(pred_move, is_moving.unsqueeze(1))
            loss = loss_bbox + loss_move
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

    torch.save(model.state_dict(), "baseball_tracker_pretrained_epoch20.pth")
    print("Model saved to baseball_tracker_pretrained_epoch20.pth")

        # Validation
    model.eval()
    with torch.no_grad():
        total, correct = 0, 0
        for images, bboxes, is_moving in val_loader:
            if images.size(0) == 0:
                continue
            _, pred_move = model(images)
            preds = (pred_move > 0.5).float().squeeze(1)
            correct += (preds == is_moving).sum().item()
            total += is_moving.size(0)
        print(f"Validation movement accuracy: {correct/total:.2f}")


In [ ]:
def train_again(train_loader, val_loader, model, start_epoch=0, end_epoch=20, model_save_path=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_bbox_fn = nn.SmoothL1Loss()
    loss_move_fn = nn.BCELoss()

    for epoch in range(start_epoch, end_epoch):
        model.train()
        for images, bboxes, is_moving in train_loader:
            if images.size(0) == 0:
                continue
            pred_bbox, pred_move = model(images)
            loss_bbox = loss_bbox_fn(pred_bbox, bboxes)
            loss_move = loss_move_fn(pred_move, is_moving.unsqueeze(1))
            loss = loss_bbox + loss_move
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

    if model_save_path is not None:
        torch.save(model.state_dict(), model_save_path)
        print(f"Model saved to {model_save_path}")

    # Validation
    model.eval()
    with torch.no_grad():
        total, correct = 0, 0
        for images, bboxes, is_moving in val_loader:
            if images.size(0) == 0:
                continue
            _, pred_move = model(images)
            preds = (pred_move > 0.5).float().squeeze(1)
            correct += (preds == is_moving).sum().item()
            total += is_moving.size(0)
        print(f"Validation movement accuracy: {correct/total:.2f}")

    return model

In [ ]:
# --- Load Model for Inference ---
def load_model(model_path="baseball_tracker_pretrained.pth"):
    model = BaseballTrackerPretrained()
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model

In [ ]:
# --- Example Usage ---
if __name__ == "__main__":
    # Replace with your actual dataset root
    train_dataset = BaseballVideos(root="/content/drive/My Drive/training_videos")
    val_dataset = BaseballVideos(root="/content/drive/My Drive/validation_videos")
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True,collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=True,collate_fn=collate_fn)
    train(train_loader, val_loader)
    #model = load_model()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 450MB/s]
/tmp/ipython-input-4109689281.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  bboxes.append(torch.tensor(box, dtype=torch.float32))


Epoch 1, Loss: 1580.0851
Epoch 2, Loss: 1557.8595
Epoch 3, Loss: 1531.9360
Epoch 4, Loss: 1503.4885
Epoch 5, Loss: 1471.0540
Epoch 6, Loss: 1437.8513
Epoch 7, Loss: 1373.9832
Epoch 8, Loss: 1351.9176
Epoch 9, Loss: 1377.2335
Epoch 10, Loss: 1235.6896
Epoch 11, Loss: 1163.4514
Epoch 12, Loss: 1086.9755
Epoch 13, Loss: 972.1372
Epoch 14, Loss: 644.0633
Epoch 15, Loss: 822.9055
Epoch 16, Loss: 730.6745
Epoch 17, Loss: 633.8712
Epoch 18, Loss: 528.7465
Epoch 19, Loss: 414.5768
Epoch 20, Loss: 317.3545
Model saved to baseball_tracker_pretrained_epoch20.pth
Validation movement accuracy: 0.01


In [ ]:
torch.save(model.state_dict(), "baseball_tracker_pretrained_epoch20.pth")

In [ ]:
# model = BaseballTrackerPretrained()
# model.load_state_dict(torch.load("baseball_tracker_pretrained.pth"))
# model.train()  # Set to training mode

In [ ]:
# if __name__ == "__main__":
#     train_dataset = BaseballVideos(root="/content/drive/MyDrive/training_videos")
#     val_dataset = BaseballVideos(root="/content/drive/MyDrive/validation_videos")
#     train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2,collate_fn=collate_fn)
#     val_loader = DataLoader(val_dataset, batch_size=4, shuffle=True, num_workers=2,collate_fn=collate_fn)

#     # model = BaseballTrackerPretrained()
#     # model.load_state_dict(torch.load("baseball_tracker_pretrained.pth"))  # Load previous weights
#     # model.train()

#     # Pass the loaded model to your train function
#     train(train_loader, val_loader)
#     #torch.save(model.state_dict(), "baseball_tracker_pretrained_epoch20.pth")  # Save updated weights

In [ ]:
# if __name__ == "__main__":
#     train_dataset = BaseballVideos(root="/content/drive/MyDrive/training_videos")
#     val_dataset = BaseballVideos(root="/content/drive/MyDrive/validation_videos")
#     train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=4,collate_fn=collate_fn)
#     val_loader = DataLoader(val_dataset, batch_size=4, shuffle=True, num_workers=4, collate_fn=collate_fn)

#     model = BaseballTrackerPretrained()
#     model.load_state_dict(torch.load("baseball_tracker_pretrained.pth"))  # Load previous weights

#     # Train for epochs 10 to 20 and save
#     model = train_again(train_loader, val_loader, model, start_epoch=10, end_epoch=20, model_save_path="baseball_tracker_pretrained_epoch20.pth")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 108MB/s]


FileNotFoundError: [Errno 2] No such file or directory: 'baseball_tracker_pretrained.pth'

## OLD

In [ ]:
# # For reading data
# import os
# import numpy as np
# from xml.dom import minidom

# from torch.utils.data import Dataset
# from torch.utils.data import DataLoader

# # For visualizing
# from torchvision.io import read_image
# from torchvision import tv_tensors
# from torchvision.transforms.v2 import functional as F

# # For model building
# import torch
# import torch.nn as nn

# # for videos
# import cv2 as cv

# class BaseballVideos(torch.utils.data.Dataset):
#     def __init__(self, root=None, transforms=None):
#         # self.root = root
#         # self.transforms = transforms
#         # load all image files, sorting them to
#         # ensure that they are aligned
#         # if root==None:
#         #     self.vids = list(sorted([i for i in os.listdir(os.path.curdir) if '.mov' in i]))
#         #     self.notes = list(sorted([i for i in os.listdir(os.path.curdir) if '.xml' in i]))
#         #     if len(self.vids)!=len(self.notes):
#         #         raise RuntimeError("Mismatch of annotation files and video files.\nPlease confirm that you have one annotation file for each video and try again.")
#         self.root = root if root is not None else os.getcwd()
#         self.transforms = transforms

#         # --- Gather files from the provided directory ---
#         self.vids = sorted([f for f in os.listdir(self.root) if f.endswith(".mov")])
#         self.notes = sorted([f for f in os.listdir(self.root) if f.endswith(".xml")])

#         imgs = []
#         notes = []
#         for i, k in zip(self.vids, self.notes):
#             i = os.path.join(self.root, i)
#             k = os.path.join(self.root, k)
#             cap = cv.VideoCapture(i)
#             note = minidom.parse(k)
#             ret = True
#             frame_count = 0
#             while ret:
#               ret, frame = cap.read()
#               if ret:
#                 frame_count += 1
#                 frame = np.moveaxis(frame, -1, 0) # Pivot image so color channels first, then H then W
#                 imgs.append(torch.from_numpy(frame))
#                 canvas_size = list(frame.shape[1:])

#             for f in range(frame_count):
#                 frame_i = [j for j in note.getElementsByTagName("box") if int(j.attributes['frame'].value)==f]
#                 boxes = []
#                 labels = []
#                 areas = []
#                 movings = []

#                 for j in frame_i:
#                     moving = j.getElementsByTagName('attribute')[0].firstChild.data=='true'

#                     xtl = float(j.attributes['xtl'].value)
#                     ytl = float(j.attributes['ytl'].value)
#                     xbr = float(j.attributes['xbr'].value)
#                     ybr = float(j.attributes['ybr'].value)
#                     box = (xtl, ytl, xbr, ybr)

#                     label = 'baseball'
#                     area = (xbr - xtl) * (ybr - ytl)

#                     boxes.append(box)
#                     labels.append(label)
#                     areas.append(area)
#                     movings.append(moving)

#                 target = {}
#                 target["boxes"] = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=canvas_size)
#                 target["labels"] = labels
#                 target["area"] = areas
#                 target["moving"] = movings

#                 notes.append(target)
#         self.imgs = imgs
#         self.notes = notes

#             # target = {}
#             # target["boxes"] = tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=F.get_size(img))
#             # target["masks"] = tv_tensors.Mask(masks)
#             # target["labels"] = labels
#             # target["image_id"] = image_id
#             # target["area"] = area
#             # target["iscrowd"] = iscrowd

#     def __len__(self):
#         return len(self.imgs)

#     def __getitem__(self, idx):

#         img = self.imgs[idx]
#         target = self.notes[idx]

#         if self.transforms is not None:
#             img, target = self.transforms(img, target)

#         return img, target

In [ ]:
import os
import numpy as np
from xml.dom import minidom
import torch
from torch.utils.data import Dataset
from torchvision import tv_tensors
from PIL import Image
import cv2 as cv

class BaseballVideos(Dataset):
    def __init__(self, root=None, transforms=None):
        self.root = root if root is not None else os.getcwd()
        self.transforms = transforms

        # Gather video and annotation files
        self.vids = sorted([f for f in os.listdir(self.root) if f.endswith(".mov")])
        self.notes = sorted([f for f in os.listdir(self.root) if f.endswith(".xml")])

        # Build a list of (video_path, xml_path, frame_number)
        self.frame_index = []
        for vid, note in zip(self.vids, self.notes):
            vid_path = os.path.join(self.root, vid)
            note_path = os.path.join(self.root, note)
            cap = cv.VideoCapture(vid_path)
            frame_count = int(cap.get(cv.CAP_PROP_FRAME_COUNT))
            for f in range(frame_count):
                self.frame_index.append((vid_path, note_path, f))
            cap.release()

    def __len__(self):
        return len(self.frame_index)

    def __getitem__(self, idx):
        vid_path, note_path, frame_num = self.frame_index[idx]

        # Load the frame on-the-fly
        cap = cv.VideoCapture(vid_path)
        cap.set(cv.CAP_PROP_POS_FRAMES, frame_num)
        ret, frame = cap.read()
        cap.release()
        if not ret:
            raise RuntimeError(f"Could not read frame {frame_num} from {vid_path}")

        # Convert frame to PIL Image (cv2 loads as BGR, PIL expects RGB)
        frame_rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)
        img = Image.fromarray(frame_rgb)

        # Parse XML for this frame
        note = minidom.parse(note_path)
        frame_i = [j for j in note.getElementsByTagName("box") if int(j.attributes['frame'].value) == frame_num]
        boxes, labels, areas, movings = [], [], [], []

        canvas_size = [img.height, img.width]
        for j in frame_i:
            moving = j.getElementsByTagName('attribute')[0].firstChild.data == 'true'
            xtl = float(j.attributes['xtl'].value)
            ytl = float(j.attributes['ytl'].value)
            xbr = float(j.attributes['xbr'].value)
            ybr = float(j.attributes['ybr'].value)
            box = (xtl, ytl, xbr, ybr)
            label = 'baseball'
            area = (xbr - xtl) * (ybr - ytl)
            boxes.append(box)
            labels.append(label)
            areas.append(area)
            movings.append(moving)

        target = {
            "boxes": tv_tensors.BoundingBoxes(boxes, format="XYXY", canvas_size=canvas_size),
            "labels": labels,
            "area": areas,
            "moving": movings
        }

        if self.transforms is not None:
            img = self.transforms(img)  # Only image is transformed here

        return img, target

In [ ]:
# -----------------------------
# 1) Basic config (keep memory low)
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 4        # small batch -> less GPU/CPU RAM
NUM_EPOCHS = 5
LR = 1e-3

In [ ]:
train_dataset = BaseballVideos(root="/content/drive/MyDrive/training_videos")

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,       # 0 keeps things simple & predictable
    #collate_fn=collate_fn
)

In [ ]:
validation_dataset = BaseballVideos(root="/content/drive/MyDrive/validation_videos")

In [ ]:
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,       # 0 keeps things simple & predictable
    #collate_fn=collate_fn
)

In [ ]:
# pretrained model
import torchvision.models as models
import torch.nn as nn

class BaseballTrackerPretrained(nn.Module):
    def __init__(self, num_outputs=4):
        super().__init__()
        # Load pretrained ResNet18
        self.backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # Remove the final classification layer
        self.backbone = nn.Sequential(*list(self.backbone.children())[:-1])  # Output: (batch, 512, 1, 1)
        self.flatten = nn.Flatten()
        self.fc_bbox = nn.Linear(512, num_outputs)  # For bounding box regression
        self.fc_move = nn.Linear(512, 1)            # For movement classification

    def forward(self, x):
        x = self.backbone(x)
        x = self.flatten(x)
        bbox = self.fc_bbox(x)
        is_moving = torch.sigmoid(self.fc_move(x))
        return bbox, is_moving

In [ ]:
import torchvision.transforms as T

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
def collate_fn(batch):
    imgs, bboxes, movings = [], [], []
    for img, target in batch:
        # Skip frames with no baseballs
        if len(target["boxes"]) == 0 or len(target["moving"]) == 0:
            continue
        # Apply transform (handles PIL Image to tensor and normalization)
        img = transform(img)
        # Use the first bounding box and movement label
        box = target["boxes"][0]
        moving = float(target["moving"][0])
        imgs.append(img)
        bboxes.append(torch.tensor(box, dtype=torch.float32))
        movings.append(torch.tensor(moving, dtype=torch.float32))
    if len(imgs) == 0:
        # Return empty tensors if all frames are skipped
        return torch.empty(0), torch.empty(0), torch.empty(0)
    imgs = torch.stack(imgs)
    bboxes = torch.stack(bboxes)
    movings = torch.stack(movings)
    return imgs, bboxes, movings

In [ ]:
# --- Training Loop ---
def train(train_loader, val_loader):
    model = BaseballTrackerPretrained()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_bbox_fn = nn.SmoothL1Loss()
    loss_move_fn = nn.BCELoss()

    for epoch in range(10):
        model.train()
        for images, bboxes, is_moving in train_loader:
            if images.size(0) == 0:
                continue
            pred_bbox, pred_move = model(images)
            loss_bbox = loss_bbox_fn(pred_bbox, bboxes)
            loss_move = loss_move_fn(pred_move, is_moving.unsqueeze(1))
            loss = loss_bbox + loss_move
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

    torch.save(model.state_dict(), "baseball_tracker_pretrained.pth")
    print("Model saved to baseball_tracker_pretrained.pth")

        # Validation
    model.eval()
    with torch.no_grad():
        total, correct = 0, 0
        for images, bboxes, is_moving in val_loader:
            if images.size(0) == 0:
                continue
            _, pred_move = model(images)
            preds = (pred_move > 0.5).float().squeeze(1)
            correct += (preds == is_moving).sum().item()
            total += is_moving.size(0)
        print(f"Validation movement accuracy: {correct/total:.2f}")


In [ ]:
# --- Load Model for Inference ---
def load_model(model_path="baseball_tracker_pretrained.pth"):
    model = BaseballTrackerPretrained()
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model

In [ ]:
# --- Example Usage ---
if __name__ == "__main__":
    # Replace with your actual dataset root
    train_dataset = BaseballVideos(root="/content/drive/MyDrive/training_videos")
    val_dataset = BaseballVideos(root="/content/drive/MyDrive/validation_videos")
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
    train(train_loader, val_loader)
    model = load_model()

/tmp/ipython-input-4109689281.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  bboxes.append(torch.tensor(box, dtype=torch.float32))


Epoch 1, Loss: 1565.9729
Epoch 2, Loss: 1523.7390
Epoch 3, Loss: 1071.0653
Epoch 4, Loss: 1415.5807
Epoch 5, Loss: 1211.1050
Epoch 6, Loss: 1240.9160
Epoch 7, Loss: 1123.8782
Epoch 8, Loss: 989.1417
Epoch 9, Loss: 907.8867
Epoch 10, Loss: 688.9261
Model saved to baseball_tracker_pretrained.pth
Validation movement accuracy: 1.00


In [ ]:
# -----------------------------
# 2) Dataset + DataLoader
# -----------------------------
train_dataset = BaseballVideos(
    root=None,              # or set to your videos folder
    #transforms=simple_transform
)

def collate_fn(batch):
    """
    batch = list of (img, target)
    """
    imgs, targets = zip(*batch)

    # Stack and convert to float in [0,1]
    imgs = torch.stack(imgs, dim=0).to(torch.float32) / 255.0

    labels = []
    for t in targets:
        # t["moving"] is a list of booleans, one per box
        if len(t["moving"]) == 0:
            labels.append(0.0)
        else:
            labels.append(1.0 if any(t["moving"]) else 0.0)
    labels = torch.tensor(labels, dtype=torch.float32)

    return imgs, labels


# train_loader = DataLoader(
#     train_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=True,
#     num_workers=0,       # 0 keeps things simple & predictable
#     collate_fn=collate_fn
# )

In [ ]:
# -----------------------------
# 3) Tiny CNN for "moving vs not moving"
# -----------------------------

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),  # [B, 64, 1, 1]
        )
        self.fc = nn.Linear(64, 1)    # output: logit for "moving"

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)     # [B, 64]
        x = self.fc(x)                # [B, 1]
        return x.squeeze(1)           # [B]

model = SmallCNN().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [ ]:


# # Optionally resize frames down to keep memory low
# def simple_transform(img, target):
#     # img: [C,H,W], uint8
#     img = img.float() / 255.0          # scale to [0,1]
#     # You can resize smaller if needed:
#     # from torchvision.transforms.v2 import functional as F
#     # img = F.resize(img, [224, 224])
#     return img, target





# -----------------------------
# 4) Training loop
# -----------------------------
def train():
    model.train()
    for epoch in range(NUM_EPOCHS):
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs = imgs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)

        epoch_loss = running_loss / len(train_dataset)
        print(f"Epoch {epoch+1}/{NUM_EPOCHS} - loss: {epoch_loss:.4f}")

    # -------------------------
    # 5) Save model + optimizer
    # -------------------------
    checkpoint = {
        "epoch": NUM_EPOCHS,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    torch.save(checkpoint, "moving_classifier4.pth")
    print("Saved checkpoint to moving_classifier4.pth")


In [ ]:
if __name__ == "__main__":
    train()

Epoch 1/5 - loss: 0.3695
Epoch 2/5 - loss: 0.2868
Epoch 3/5 - loss: 0.2982
Epoch 4/5 - loss: 0.2868
Epoch 5/5 - loss: 0.3096
Saved checkpoint to moving_classifier.pth


In [ ]:
# For reloading
import torch
#from train_moving import SmallCNN  # or redefine the same class here

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SmallCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

checkpoint = torch.load("moving_classifier3.pth", map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
start_epoch = checkpoint["epoch"]

model.eval()  # ready for inference
print(f"Restored model from epoch {start_epoch}")


Restored model from epoch 5


In [ ]:
# 2nd round of training
if __name__ == "__main__":
    train()

Epoch 1/5 - loss: 0.4086
Epoch 2/5 - loss: 0.3954
Epoch 3/5 - loss: 0.3959
Epoch 4/5 - loss: 0.4012
Epoch 5/5 - loss: 0.3931
Saved checkpoint to moving_classifier2.pth


In [ ]:
# 3rd round of training
if __name__ == "__main__":
    train()

Epoch 1/5 - loss: 0.3939
Epoch 2/5 - loss: 0.3957
Epoch 3/5 - loss: 0.3957
Epoch 4/5 - loss: 0.3933
Epoch 5/5 - loss: 0.3945
Saved checkpoint to moving_classifier3.pth


In [ ]:
# 4th round of training
if __name__ == "__main__":
    train()

Epoch 1/5 - loss: 0.3853
Epoch 2/5 - loss: 0.3980
Epoch 3/5 - loss: 0.3904
Epoch 4/5 - loss: 0.3865
Epoch 5/5 - loss: 0.3837
Saved checkpoint to moving_classifier4.pth


### Take 2

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

In [ ]:
#import data loader

In [ ]:


# --- Model Definition ---
class BaseballTracker(nn.Module):
    def __init__(self, input_size=(3, 64, 64)):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, *input_size)
            feat = self.features(dummy)
            feat_size = feat.view(1, -1).shape[1]
        self.fc_bbox = nn.Linear(feat_size, 4)
        self.fc_move = nn.Linear(feat_size, 1)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        bbox = self.fc_bbox(x)
        is_moving = torch.sigmoid(self.fc_move(x))
        return bbox, is_moving

# --- Collate Function for DataLoader ---
def collate_fn(batch):
    imgs = []
    bboxes = []
    movings = []
    for img, target in batch:
        if len(target["boxes"]) == 0:
          continue
        box = target["boxes"][0]  # Should be a sequence of 4 numbers
        imgs.append(img.float() / 255.0)
        bboxes.append(torch.tensor(box, dtype=torch.float32))
        movings.append(torch.tensor(float(target["moving"][0]), dtype=torch.float32))
    if len(imgs) == 0:
        return torch.empty(0), torch.empty(0), torch.empty(0)
    imgs = torch.stack(imgs)
    bboxes = torch.stack(bboxes)
    movings = torch.stack(movings)
    return imgs, bboxes, movings

# --- Training Loop ---
def train(train_loader, val_loader, input_size=(3, 64, 64)):
    model = BaseballTracker(input_size=input_size)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_bbox_fn = nn.SmoothL1Loss()
    loss_move_fn = nn.BCELoss()

    for epoch in range(10):
        model.train()
        for images, bboxes, is_moving in train_loader:
            if images.size(0) == 0:
              continue # skip empty batch
            pred_bbox, pred_move = model(images)
            loss_bbox = loss_bbox_fn(pred_bbox, bboxes)
            loss_move = loss_move_fn(pred_move, is_moving.unsqueeze(1))
            loss = loss_bbox + loss_move
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

    # Save the model
    torch.save(model.state_dict(), "baseball_tracker.pth")
    print("Model saved to baseball_tracker.pth")

    # Validation
    model.eval()
    with torch.no_grad():
        total, correct = 0, 0
        for images, bboxes, is_moving in val_loader:
            _, pred_move = model(images)
            preds = (pred_move > 0.5).float().squeeze(1)
            correct += (preds == is_moving).sum().item()
            total += is_moving.size(0)
        print(f"Validation movement accuracy: {correct/total:.2f}")

# --- Loading the Model for Inference ---
def load_model(input_size=(3, 64, 64), model_path="baseball_tracker.pth"):
    model = BaseballTracker(input_size=input_size)
    model.load_state_dict(torch.load(model_path))
    model.eval()
    return model



In [ ]:
from torchvision.transforms.v2 import Resize

In [ ]:
transform = Resize((64,64))

In [ ]:
train_dataset = BaseballVideos(root="/content/drive/MyDrive/training_videos", transforms=transform)

In [ ]:
val_dataset = BaseballVideos(root="/content/drive/MyDrive/validation_videos",transforms=transform)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,num_workers=0, collate_fn=collate_fn)


In [ ]:
val_loader = DataLoader(val_dataset, batch_size=4, num_workers=0, collate_fn=collate_fn)

In [ ]:
train(train_loader, val_loader)

RuntimeError: The size of tensor a (0) must match the size of tensor b (4) at non-singleton dimension 1

In [ ]:
# --- Example Usage ---
if __name__ == "__main__":
    # Replace with your actual dataset root
    # train_dataset = BaseballVideos(root="/content/drive/MyDrive/training_videos")
    # val_dataset = BaseballVideos(root="/content/drive/MyDrive/validation_videos")
    train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,num_workers=0, collate_fn=collate_fn)
    val_loader = DataLoader(val_dataset, batch_size=4, num_workers=0, collate_fn=collate_fn)
    train(train_loader, val_loader)
    model = load_model()
    # Now you can use model for inference

RuntimeError: The size of tensor a (0) must match the size of tensor b (4) at non-singleton dimension 1

In [ ]:
# Example usage (replace with your actual data loaders)
# train_loader = ...
# val_loader = ...
# train(train_loader, val_loader)
# model = load_model()

NameError: name 'val_loader' is not defined